# Recovery Score Analysis

## Part 1: Linear Regression Plan

For the first part of the analysis, we will fit a linear regression model with:

- **X variables:** age, gender, workout type, workout time of the day
- **Y variable:** recovery score

This section will include data preparation, encoding of categorical predictors, and model fitting to evaluate how these features relate to recovery score.


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

In [16]:
df = pd.read_csv('../data/whoop_fitness_dataset_100k.csv')

# Replace NaN in workout_time_of_day with a meaningful category
df['workout_time_of_day'] = df['workout_time_of_day'].fillna('No_Workout')

# Categorical columns
categorical_onehot = ['gender', 'primary_sport', 'activity_type', 'workout_time_of_day']
ordinal_cols = ['fitness_level']

# Ordinal mapping for fitness level
fitness_order = [['Beginner', 'Intermediate', 'Advanced', 'Elite']]

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_onehot),
        ('ordinal', OrdinalEncoder(categories=fitness_order), ordinal_cols)
    ],
    remainder='passthrough'   # keeps numeric columns like age, sleep_efficiency
)


In [9]:
X_cols = ['age',
          'gender',
          'fitness_level',
          'primary_sport',
          'activity_type',
          'workout_time_of_day']

y_col = 'recovery_score'
df[X_cols].head()


,age,gender,fitness_level,primary_sport,activity_type,workout_time_of_day
0,56,Female,Beginner,Weight Training,Rest Day,No_Workout
1,56,Female,Beginner,Weight Training,Weight Training,Evening
2,56,Female,Beginner,Weight Training,Rest Day,No_Workout
3,56,Female,Beginner,Weight Training,Rest Day,No_Workout
4,56,Female,Beginner,Weight Training,Weight Training,Evening


In [13]:
df[X_cols + [y_col]].info()


<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   age                  100000 non-null  int64  
 1   gender               100000 non-null  str    
 2   fitness_level        100000 non-null  str    
 3   primary_sport        100000 non-null  str    
 4   activity_type        100000 non-null  str    
 5   workout_time_of_day  100000 non-null  str    
 6   recovery_score       100000 non-null  float64
dtypes: float64(1), int64(1), str(5)
memory usage: 5.3 MB


In [18]:
print("Unique fitness_level values:")
print(df['fitness_level'].unique())
print("\nValue counts:")
print(df['fitness_level'].value_counts())

Unique fitness_level values:
<StringArray>
['Beginner', 'Elite', 'Intermediate', 'Advanced']
Length: 4, dtype: str

Value counts:
fitness_level
Intermediate    40702
Advanced        27447
Beginner        19065
Elite           12786
Name: count, dtype: int64


In [19]:
X = df[['age',
        'gender',
        'fitness_level',
        'primary_sport',
        'activity_type',
        'workout_time_of_day']]

y = df['recovery_score']

# -----------------------------
# 2. Build the pipeline
# -----------------------------
model = Pipeline(steps=[
    ('preprocess', preprocessor),   # your encoder from earlier
    ('regressor', LinearRegression())
])

# -----------------------------
# 3. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 4. Fit the model
# -----------------------------
model.fit(X_train, y_train)

# -----------------------------
# 5. Evaluate
# -----------------------------
r2 = model.score(X_test, y_test)
print("R² score:", r2)

R² score: 0.011540467689753808
